# KOZ-3 · v22 push, phase 1 (local RTX 4080 SUPER + 32 cores)

Runs inside **`Codex Work/dinov3_wt`**, the checkout where Codex's exact-v21/v22 satellite tests already work with all
out-of-fold caches. Only Claude's files are updated there (`git checkout <branch> -- <files>`); Codex's uncommitted work in
that folder is left untouched.

| # | test | why | expected |
|---|---|---|---|
| A | DINOv3 hillfort localiser + gate in exact v21/v22 satellite decoding | Codex's 28 Oct run crashed on a column-name clash (`y3857`), so it was never measured; hillforts = the 5–10k m² recall-0 gap | +0.005 … +0.02 |
| B | hillfort-screenshot kurgan gate **with scene protection** | 28 Oct: best fixed +0.018, robust k*=16, P=1.00, but fold 0 regressed (Kurmentu / Izborsk) | +0.005 … +0.015 |
| C | Satlas Swin-B (0.5–2 m aerial-pretrained) fine-tuned on GPU, 8-way TTA, as a 7th satellite member | new architecture family (all 6 members are DETR / heatmap variants) | 0 … +0.01 |

**Acceptance (unchanged):** nested honest > deployed, no outer fold regresses, k* ≥ 3, bootstrap P(gain>0) ≥ 0.8.
A and B take ~20–60 min each on CPU; C ~2–4 h on the GPU. Run top to bottom; the last cell prints everything to paste back.

In [ ]:
import os, subprocess
MAIN = "/home/nathan/Expedition Competition"
WT = os.path.join(MAIN, "Codex Work", "dinov3_wt")
os.chdir(WT); print(os.getcwd())
FILES = ["scripts/sp_hillfort_dinov3_loc_oof.py", "scripts/sp_hillfort_scene_kurgan_gate_oof.py", "scripts/robust_accept.py", "scripts/sp_add_member_oof.py",
         "experiments/claude/outputs/hf_dinov3_loc_points_oof.csv", "experiments/claude/outputs/oof_det_cpu_ens_cands.parquet", "experiments/claude/sp_cnn", "experiments/claude/sp_dense",
         "experiments/claude/hf_scene_cls.py"]
subprocess.run(["git", "fetch", "origin", "feat/physics-gates"], check=True)
subprocess.run(["git", "checkout", "origin/feat/physics-gates", "--"] + FILES, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline", "origin/feat/physics-gates"], capture_output=True, text=True).stdout)
# the caches these tests read (must already exist here, as in Codex's 28 Oct runs)
for p in ["work/spor/oof_scene_cls.parquet", "data/labels_3857.parquet", "kaggle_import/out/koz3-sp-det/sp_runs/sp_dfine_l/folds.json"]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)
os.makedirs("logs", exist_ok=True)

## A · DINOv3 hillfort localiser + gate (CPU, ~30–60 min)

In [ ]:
!python scripts/sp_hillfort_dinov3_loc_oof.py 2>&1 | grep -v -i "warning\|concat" | tee logs/claude_A_loc_gate_v22.log | tail -45

## B · hillfort-screenshot kurgan gate with scene protection (CPU, ~30–60 min)

In [ ]:
!python scripts/sp_hillfort_scene_kurgan_gate_oof.py 2>&1 | grep -v -i "warning\|concat" | tee logs/claude_B_kurgan_gate_protect_v22.log | tail -50

## B2 · CPU satellite ensemble (ResNet-18 + DINOv3 heads, d4) as a 7th member (CPU, ~20 min)

In [ ]:
import shutil; shutil.copy("experiments/claude/outputs/oof_det_cpu_ens_cands.parquet", "work/spor/oof_det_cpu_ens_cands.parquet")
!python scripts/sp_add_member_oof.py cpu_ens 2>&1 | grep -v -i "warn\|concat" | tee logs/claude_B2_add_cpu_ens.log | tail -15

## C · Satlas Swin-B on the GPU as a 7th satellite member (~2–4 h)
1. 0.5 m image cache for the satellite scenes (CPU, ~10 min) · 2. fine-tune 4 geographic folds (checkpoints every 250 steps;
re-running resumes) · 3. out-of-fold 8-way TTA · 4. export as a fusion member · 5. exact v21/v22 satellite score with 6 vs 7 members.

In [ ]:
!python experiments/claude/sp_cnn/cache.py 2>&1 | tail -2

In [ ]:
!CNN_DEVICE=cuda CNN_BS=24 CNN_LR=2e-4 python experiments/claude/sp_cnn/train.py 6000 0,1,2,3 satlas_swinb 2>&1 | grep --line-buffered -v -i warn | tee logs/claude_C_satlas_train.log | grep --line-buffered -E "device|scenes|it [0-9]*000 |done|Error|Trace" 

In [ ]:
!CNN_DEVICE=cuda python experiments/claude/sp_cnn/predict_d4.py satlas_swinb satlas_swinb_d4 satlas_swinb 0 2>&1 | tail -5
!PEAK_K=7 python experiments/claude/sp_cnn/export_member.py work/sp_cnn/oof_satlas_swinb_d4 satlas 2>&1 | tail -1

In [ ]:
# exact v21/v22 satellite decoding with the new member at weight 0 (= deployed) / 0.5 / 1, nested + robust
!python scripts/sp_add_member_oof.py satlas 2>&1 | grep -v -i "warn\|concat" | tee logs/claude_C_add_satlas.log | tail -15
# Satlas alone in the CPU-pipeline metric (comparable with the CPU ResNet-18 0.226 / CPU ensemble 0.339)
!HF_SCORE=mean HF_POINTS=experiments/claude/outputs/hf_dinov3_loc_points_oof.csv PEAK_K=7 python experiments/claude/sp_dense/evaluate.py work/sp_cnn/oof_satlas_swinb_d4 2>&1 | grep -A6 honest | tee logs/claude_C_satlas_alone.log

## Results to paste back

In [ ]:
for f in ["claude_A_loc_gate_v22", "claude_B_kurgan_gate_protect_v22", "claude_B2_add_cpu_ens", "claude_C_add_satlas", "claude_C_satlas_alone"]:
    p = f"logs/{f}.log"
    print("=" * 30, f); print(open(p).read()[-6000:] if os.path.exists(p) else "(not run)")